# Tutorial 6.1: scSLAT Registration of P22 Mouse Brain Adjacent RNA and ATAC Sections

This tutorial uses S3 RNA and S2 ATAC sections from the P22 mouse brain spatial epigenome-transcriptome dataset reported by [Zhang et al.](https://doi.org/10.1038/s41586-023-05795-1). The study jointly profiles chromatin accessibility and gene expression at near-single-cell resolution; the processed P22 dataset is publicly available through the [AtlasXplore Fan dataset](https://web.atlasxomics.com/visualization/Fan).

[SLAT](https://doi.org/10.1038/s41467-023-43105-5) provides the heterogeneous spatial registration used here. Its [official code](https://github.com/gao-lab/SLAT) and [documentation](https://slat.readthedocs.io/) describe the scSLAT implementation. High-confidence RNA-to-ATAC correspondences are projected into the S2 coordinate system to create the partially observed RNA-on-ATAC object analysed by PRISM in Tutorial 6.2.

This preserves the real adjacent-section setting: only cross-modality correspondences supported after confidence filtering are retained for the downstream incomplete-registration analysis.


### GLUE preparation

scSLAT begins from precomputed GLUE embeddings, which place RNA and ATAC profiles in a shared feature space before spatial registration. GLUE preparation is performed externally; this tutorial directly loads the prepared AnnData inputs, following the [GLUE documentation](https://scglue.readthedocs.io/) and [scSLAT tutorial](https://slat.readthedocs.io/en/latest/tutorials/basic_usage.html).


In [ ]:
# 0. Environment and imports
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
from scSLAT.model import Cal_Spatial_Net, load_anndatas, run_SLAT, spatial_match
from scSLAT.viz import match_3D_multi

plt.rcParams.update({"font.size": 11, "axes.linewidth": 1.0, "pdf.fonttype": 42})

In [ ]:
# Load prepared inputs
DATASET_DIR = Path("Datasets") / "P22 mouse brain_adjacent sections"
adata1 = sc.read_h5ad(DATASET_DIR / "S3_adata_RNA.h5ad")
adata2 = sc.read_h5ad(DATASET_DIR / "S2_adata_ATAC.h5ad")

Both AnnData objects store the precomputed cross-modality GLUE representation in `obsm["X_glue"]`, which scSLAT uses for registration. Their `obs["mclust"]` columns store the mclust cluster label for each spatial location; these labels are used only to colour and display spatial domains in the subsequent 3D alignment visualization and do not affect the alignment procedure.

In [ ]:
adata1

In [ ]:
# Build spatial graphs
Cal_Spatial_Net(adata1, k_cutoff=20, model="KNN")
Cal_Spatial_Net(adata2, k_cutoff=20, model="KNN")

In [ ]:
# Run SLAT with the precomputed GLUE representation
edges, features = load_anndatas([adata1, adata2], feature="glue")
embd0, embd1, elapsed_time = run_SLAT(features, edges, LGCN_layer=6, hidden_size=4096)

In [ ]:
# Match ATAC locations to RNA: Store SLAT embeddings and match ATAC to RNA
def to_numpy_embedding(embedding):
    return embedding.detach().cpu().numpy() if torch.is_tensor(embedding) else np.asarray(embedding)

adata1.obsm["X_slat"] = to_numpy_embedding(embd0).astype(np.float32, copy=False)
adata2.obsm["X_slat"] = to_numpy_embedding(embd1).astype(np.float32, copy=False)
best, index, distance = spatial_match([embd0, embd1], adatas=[adata1, adata2], reorder=False,
                                      smooth_range=200)
label_key = "mclust"
adata1_df = pd.DataFrame({"index": np.arange(embd0.shape[0]), "x": adata1.obsm["spatial"][:, 0],
                          "y": adata1.obsm["spatial"][:, 1], label_key: adata1.obs[label_key]})
adata2_df = pd.DataFrame({"index": np.arange(embd1.shape[0]), "x": adata2.obsm["spatial"][:, 0],
                          "y": adata2.obsm["spatial"][:, 1], label_key: adata2.obs[label_key]})
matching = np.array([np.arange(index.shape[0]), best])
best_match = distance[:, 0]

In [ ]:
# Inspect matching similarity
MATCH_THRESHOLD = 0.85
fig, ax = plt.subplots(figsize=(5, 3.6))
n, bins, patches = ax.hist(distance[:, 0], bins=100, edgecolor="#8C9093", alpha=0.7)
for left, right, patch in zip(bins[:-1], bins[1:], patches):
    if 0.5 * (left + right) < MATCH_THRESHOLD:
        patch.set_facecolor("#8C9093")
        patch.set_edgecolor("#8C9093")
    else:
        patch.set_facecolor("#3990CE")
        patch.set_edgecolor("#3990CE")
ax.vlines(MATCH_THRESHOLD, 0, n.max(), color="#d62728", linestyles="dotted",
          label=f"Threshold = {MATCH_THRESHOLD:.2f}")
ax.set_xlabel("Cosine similarity")
ax.set_ylabel("ATAC cells")
ax.grid(False)
ax.legend(frameon=False)
plt.show()

The score distribution is used to choose the threshold that separates retained registrations from low-confidence correspondences.


### Retain high-confidence registrations

Only correspondences above the selected score threshold are treated as retained RNA-ATAC pairs.


In [ ]:
# Filter high-confidence pairs
matching_filter = matching[:, distance[:, 0] > MATCH_THRESHOLD]
print(f"High-confidence pairs: {matching_filter.shape[1]:,} / {adata2.n_obs:,}")

In [ ]:
# Inspect RNA-ATAC registration in 3D
multi_align = match_3D_multi(adata1_df, adata2_df, matching_filter, meta=label_key, rotate=["y", "y"],
                             scale_coordinate=True, subsample_size=300)
multi_align.draw_3D([5, 5], line_width=0.5, point_size=[1, 0.5], hide_axis=True, show_error=True)

The 3D display provides a qualitative check that retained RNA-ATAC correspondences align across the adjacent sections.


### Build registered RNA-on-ATAC data

The output follows the S2 ATAC coordinate order. ATAC locations without a retained RNA partner receive zero RNA values and `missing="0"`, producing the real incomplete target object for Tutorial 6.2.


In [ ]:
# Create and save the ATAC-ordered registered RNA object
# Rows follow original S2 ATAC locations; unregistered locations receive zero RNA values.
target_idx = matching_filter[0].astype(int)
align_idx = matching_filter[1].astype(int)
matched_expression = adata1[align_idx, :].X
if not sp.issparse(matched_expression):
    matched_expression = sp.csr_matrix(np.asarray(matched_expression, dtype=np.float32))
else:
    matched_expression = matched_expression.tocsr().astype(np.float32)
matched_expression = matched_expression.tocoo()
full_X = sp.csr_matrix((matched_expression.data,
    (target_idx[matched_expression.row], matched_expression.col)), shape=(adata2.n_obs, adata1.n_vars),
    dtype=np.float32)
full_obs = adata2.obs.copy()
full_obs["missing"] = "0"
full_obs["matched_RNA_barcode"] = "NA"
full_obs["SLAT_score"] = np.nan
full_obs.loc[adata2.obs_names[target_idx], "missing"] = "1"
full_obs.loc[adata2.obs_names[target_idx], "matched_RNA_barcode"] = adata1.obs_names.to_numpy()[align_idx]
full_obs.loc[adata2.obs_names[target_idx], "SLAT_score"] = best_match[target_idx]
new_adata_full = ad.AnnData(X=full_X, obs=full_obs, var=adata1.var.copy())
new_adata_full.obs_names = adata2.obs_names.copy()
new_adata_full.obsm["spatial"] = adata2.obsm["spatial"].copy()
new_adata_full.uns["registration"] = {"method": "scSLAT", "reference": "S3 RNA", "query": "S2 ATAC",
                                       "similarity_threshold": MATCH_THRESHOLD}
new_adata_full.write_h5ad(DATASET_DIR / "S3_adata_RNA_reg.h5ad")

In [ ]:
# Display real registration missingness and transferred RNA expression
new_adata_full.uns["missing_colors"] = ["#d62728", "#1f77b4"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
sc.pl.embedding(new_adata_full, basis="spatial", color="missing", ax=axes[0], title="Registration status",
    size=21, legend_loc="right margin", show=False)
legend = axes[0].get_legend()
for text, label in zip(legend.get_texts(), ["Unregistered", "Registered"]):
    text.set_text(label)
for handle in getattr(legend, "legend_handles", getattr(legend, "legendHandles", [])):
    if hasattr(handle, "set_sizes"):
        handle.set_sizes([20.0])
sc.pl.embedding(new_adata_full, basis="spatial", color="Atp1b1", ax=axes[1],
    title="Representative gene: Atp1b1", size=21, cmap="viridis",
    mask_obs=new_adata_full.obs["missing"].astype(str) == "1", na_color="#B8B8B8", show=False)
for ax in axes:
    ax.set_xlabel("spatial1")
    ax.set_ylabel("spatial2")
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
plt.subplots_adjust(left=0.05, right=0.98, bottom=0.15, top=0.85, wspace=0.55)
plt.show()

The final panels show retained and unregistered S2 locations together with an example transferred RNA feature.
